# ExoHunter — Student Exercises

**Mission: analyze a mystery star and discover its planets**

In the exploratory notebook you followed a guided tour of TOI-700.
Now it's your turn. You've been assigned a **real TESS target**:

> **L 98-59** (TIC 261136679) — an ultracool M-dwarf at 10.6 parsecs

This star is known to host **at least 3 transiting planets**, but
pretend you don't know that. Your job is to analyze the data from
scratch, discover the planets, validate them, and write up your
findings — exactly as a researcher would.

### Rules

- Fill in every cell marked with `# TODO`
- Don't look up the answer — discover it from the data
- Run each cell and interpret the output before moving on
- Write your observations in the markdown cells provided

### Grading rubric

| Criterion | Points |
|-----------|--------|
| All code cells execute without errors | 20 |
| Correctly identifies all detectable planets | 25 |
| Validation and cross-match interpretation | 15 |
| Noise experiment: clear plot + correct conclusion | 20 |
| Written analysis quality (observations cells) | 20 |

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from exohunter.ingestion.downloader import download_lightcurve
from exohunter.preprocessing.pipeline import preprocess_single
from exohunter.detection.bls import run_bls_lightkurve, run_iterative_bls, TransitCandidate
from exohunter.detection.model import transit_model, phase_fold, bin_phase_curve
from exohunter.detection.validator import validate_candidate
from exohunter.catalog.crossmatch import crossmatch_candidate
from exohunter.catalog.candidates import compute_score

print("Imports ready.")

---

## Exercise 1: Download and explore the light curve

Download all available TESS sectors for **TIC 261136679** and
plot the raw light curve. Answer the questions below.

In [ ]:
# TODO: Set the TIC ID and download the light curve
TIC_ID = "..."  # Fill in the correct TIC ID

light_curve = download_lightcurve(TIC_ID)

time_raw = np.array(light_curve.time.value, dtype=np.float64)
flux_raw = np.array(light_curve.flux.value, dtype=np.float64)

print(f"Cadences: {len(time_raw)}")
print(f"Time span: {time_raw[-1] - time_raw[0]:.1f} days")
print(f"Approx. sectors: {(time_raw[-1] - time_raw[0]) / 27.4:.0f}")

In [ ]:
# TODO: Plot the raw light curve using Plotly
# Hint: use go.Scattergl for large datasets, template="plotly_dark"
# Include: title, axis labels, range slider

fig = go.Figure()
# ... your code here ...
fig.show()

### Your observations

*TODO: Write 2-3 sentences about what you see in the raw light curve.
Are there obvious gaps? Outliers? Any visible periodic dips?*

...

---

## Exercise 2: Preprocess the data

Run the preprocessing pipeline and compare the result to the raw data.
The pipeline removes NaN values, clips outliers, normalizes flux to 1.0,
and removes long-period stellar variability.

In [ ]:
# TODO: Run preprocess_single on the light curve
processed = ...  # Fill in

time_proc = processed.time
flux_proc = processed.flux

print(f"Before: {len(time_raw)} cadences")
print(f"After:  {len(time_proc)} cadences")
print(f"Removed: {len(time_raw) - len(time_proc)} ({100*(1 - len(time_proc)/len(time_raw)):.1f}%)")
print(f"CDPP:    {processed.cdpp:.1f} ppm")

In [ ]:
# TODO: Create a 2-row subplot comparing raw vs processed
# Hint: use make_subplots(rows=2, cols=1, shared_xaxes=True)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Raw", "Processed"])
# ... your code here ...
fig.show()

---

## Exercise 3: Run BLS and interpret the periodogram

L 98-59's planets have short periods (all < 8 days). Choose an
appropriate BLS search range and run the detection.

**Think carefully**: what should `min_period` and `max_period` be?
Too narrow and you miss planets. Too wide and you waste compute time.

In [ ]:
# TODO: Choose the BLS period range and run the search
# Think: this is a short-period system. What range makes sense?

lc_for_bls = processed.to_lightcurve()

min_period = ...  # TODO: choose (days)
max_period = ...  # TODO: choose (days)
num_periods = 10000

period_grid = np.linspace(min_period, max_period, num_periods)
periodogram = lc_for_bls.to_periodogram(
    method="bls", period=period_grid, frequency_factor=500,
)

best_period = float(periodogram.period_at_max_power.value)
best_depth = float(periodogram.depth_at_max_power.value)

print(f"Strongest signal: P = {best_period:.4f} days")
print(f"Transit depth:    {best_depth * 100:.4f}%")

In [ ]:
# TODO: Plot the BLS periodogram
# Mark the strongest peak with a vertical line
# Question: do you see other peaks? What might they be?

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=periodogram.period.value, y=periodogram.power.value,
    mode="lines", line=dict(color="deepskyblue", width=1),
))
# TODO: add a vertical line at best_period
# Hint: fig.add_vline(x=..., line=dict(color="red", dash="dash"))

fig.update_layout(
    template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    title="BLS Periodogram",
    xaxis_title="Period (days)", yaxis_title="BLS Power",
    height=400,
)
fig.show()

### Your observations

*TODO: How many peaks do you see in the periodogram? Are any of them
at exact multiples (2×, 3×) or fractions (1/2, 1/3) of the main peak?
What does this tell you about the system?*

...

---

## Exercise 4: Phase-fold and visualize the strongest transit

Fold the light curve at the best BLS period. You should see a
clear transit dip near phase = 0.

In [ ]:
# TODO: Phase-fold at the best period and plot
# Use phase_fold() and bin_phase_curve() from exohunter.detection.model

best_t0 = float(periodogram.transit_time_at_max_power.value)
best_dur = float(periodogram.duration_at_max_power.value)

phase, flux_folded = phase_fold(time_proc, flux_proc, best_period, best_t0)
centers, means, stds = bin_phase_curve(phase, flux_folded, n_bins=200)

fig = go.Figure()

# TODO: Add two traces:
# 1. Raw phase-folded data (Scattergl, small semi-transparent markers)
# 2. Binned data (Scatter with error bars)
# ... your code here ...

fig.update_layout(
    template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    title=f"Phase-Folded at P = {best_period:.4f} d",
    xaxis_title="Orbital Phase", yaxis_title="Normalized Flux",
    xaxis_range=[-0.2, 0.2],  # zoom into transit
    height=400,
)
fig.show()

### Your observations

*TODO: Describe the transit shape. Is it box-like or V-shaped?
Estimate the depth by eye (in percent). How does this compare to
the BLS-reported depth?*

...

---

## Exercise 5: Find ALL the planets

Use `run_iterative_bls()` to search for multiple planets.
The function subtracts each detected transit and re-runs BLS.

In [ ]:
# TODO: Run the iterative multi-planet search
# Choose appropriate parameters:
#   - min_period, max_period: same as your BLS range
#   - max_planets: how many do you expect? (start with 5)
#   - min_snr: use 0.0 for synthetic-compatible threshold

candidates = run_iterative_bls(
    lc_for_bls,
    tic_id=TIC_ID,
    min_period=...,   # TODO
    max_period=...,   # TODO
    max_planets=...,  # TODO
    min_snr=0.0,
)

print(f"Found {len(candidates)} candidate(s):\n")
for i, c in enumerate(candidates):
    letter = chr(ord('b') + i)
    print(f"  Planet {letter}: P = {c.period:.4f} d, depth = {c.depth*100:.4f}%")

In [ ]:
# TODO: Create a multi-panel phase-fold plot showing each detected planet
# For each candidate, phase-fold the data at that planet's period and plot.
#
# Hint: for the 2nd and 3rd planets, you should subtract the previous
# planets' transit models before phase-folding. Use:
#   model = transit_model(time, period, epoch, duration, depth)
#   residual_flux = flux - (model - 1.0)

n_found = len(candidates)
if n_found > 0:
    fig = make_subplots(
        rows=1, cols=n_found,
        subplot_titles=[f"Planet {chr(ord('b')+i)} (P={c.period:.3f} d)"
                        for i, c in enumerate(candidates)],
    )

    residual = flux_proc.copy()
    colors = ["#ff6b6b", "#4ecdc4", "#ffd93d", "#6bcb77", "#9b59b6"]

    for i, c in enumerate(candidates):
        # TODO: phase-fold residual at this planet's period
        # TODO: bin the phase-folded data
        # TODO: add a trace to fig (row=1, col=i+1)
        # TODO: subtract this planet's model from residual for the next iteration
        pass  # Replace with your code

    fig.update_layout(
        template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        title="All Detected Planets — Phase-Folded",
        height=400, showlegend=False,
    )
    for i in range(n_found):
        fig.update_xaxes(range=[-0.15, 0.15], row=1, col=i+1)
    fig.show()

### Your observations

*TODO: How many planets did you find? List their periods.
Are any of the detected periods harmonics of each other?
Which planet has the deepest transit? The shallowest?*

...

---

## Exercise 6: Validate and cross-match each candidate

For each detected planet, run the 6-criterion validator and
cross-match against the TOI catalog. Discover whether these
are known planets or new candidates.

In [ ]:
# TODO: For each candidate, run validate_candidate and crossmatch_candidate
# Print a summary table with: planet letter, period, depth, validation result,
# cross-match classification, and catalog match name (if any).

print(f"{'Planet':<8} {'Period (d)':<12} {'Depth (%)':<10} {'Valid?':<8} {'Cross-match':<16} {'Catalog'}")
print("-" * 75)

for i, c in enumerate(candidates):
    letter = chr(ord('b') + i)

    # TODO: run validate_candidate (pass time_proc and flux_proc)
    validation = ...  # Fill in

    # TODO: run crossmatch_candidate
    xmatch = ...  # Fill in

    valid_str = "YES" if validation.is_valid else "NO"
    print(f"  {letter:<6} {c.period:<12.4f} {c.depth*100:<10.4f} {valid_str:<8} "
          f"{xmatch.match_class.value:<16} {xmatch.catalog_name}")

### Your observations

*TODO: Were all your candidates classified as KNOWN_MATCH? If any
were classified as HARMONIC or NEW_CANDIDATE, explain why.
Which planets were validated and which were rejected? What criteria
did the rejected ones fail?*

...

---

## Exercise 7: Detection limits — how faint can we go?

This is the most important experiment. You'll inject a **synthetic
planet** into real noise and measure how small a planet ExoHunter can
detect.

The question: **what is the minimum transit depth that BLS can recover
at the noise level of this star?**

Protocol:
1. Take the preprocessed light curve (which has real TESS noise)
2. Subtract all detected planets (so only noise remains)
3. Inject a synthetic transit at a known period and depth
4. Run BLS and check if it recovers the injected period
5. Repeat for decreasing depths until BLS fails
6. Plot: recovery rate vs. injected depth

In [ ]:
# Step 1: Create a "noise-only" light curve by subtracting all detected planets
noise_flux = flux_proc.copy()
for c in candidates:
    model = transit_model(time_proc, c.period, c.epoch, c.duration, c.depth)
    noise_flux = noise_flux - (model - 1.0)

noise_std = np.std(noise_flux)
print(f"Residual noise level: {noise_std*1e6:.0f} ppm ({noise_std*100:.4f}%)")
print(f"This is the noise floor — transits shallower than ~3× this are undetectable.")
print(f"Predicted detection limit: ~{3*noise_std*100:.4f}% depth")

In [ ]:
# Step 2: Injection-recovery experiment
# TODO: Complete the experiment loop
#
# For each trial depth:
#   1. Start with noise_flux (noise-only residuals)
#   2. Inject a transit: subtract `depth` from flux where in-transit
#      Use: transit_model(time_proc, period=4.0, epoch=2.0, duration=0.08, depth=trial_depth)
#   3. Build a LightCurve, run BLS
#   4. Check if BLS found a period within 0.1 d of the injected 4.0 d
#   5. Record success or failure

from lightkurve import LightCurve

injected_period = 4.0  # days
injected_epoch = 2.0
injected_duration = 0.08  # ~2 hours

# Test depths from large (easy) to tiny (impossible)
trial_depths = [0.005, 0.003, 0.002, 0.0015, 0.001, 0.0008, 0.0005, 0.0003, 0.0001]
recovered = []

for depth in trial_depths:
    # Inject the synthetic transit into the noise residuals
    injected_model = transit_model(
        time_proc, injected_period, injected_epoch, injected_duration, depth,
    )
    test_flux = noise_flux + (injected_model - 1.0)  # add the dip

    # TODO: create a LightCurve from time_proc and test_flux
    test_lc = ...  # Fill in: LightCurve(time=..., flux=...)

    # TODO: run BLS on test_lc
    result = ...  # Fill in: run_bls_lightkurve(test_lc, ...)

    # Check if the injected period was recovered
    if result is not None and abs(result.period - injected_period) < 0.1:
        recovered.append(True)
        status = "RECOVERED"
    else:
        recovered.append(False)
        found_p = f"{result.period:.2f}" if result else "none"
        status = f"MISSED (found P={found_p})"

    print(f"  depth={depth*100:.4f}%  ({depth*1e6:.0f} ppm)  →  {status}")

In [ ]:
# TODO: Plot the injection-recovery results
# X-axis: injected depth (ppm), Y-axis: recovered (1) or not (0)
# Use a bar chart or scatter plot. Mark the detection threshold.

depths_ppm = [d * 1e6 for d in trial_depths]
recovery_int = [1 if r else 0 for r in recovered]

fig = go.Figure()
# ... your plotting code here ...
# Hint: use go.Bar with marker_color based on recovered (green/red)

fig.update_layout(
    template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    title="Injection-Recovery: Detection Limit of ExoHunter",
    xaxis_title="Injected Depth (ppm)", yaxis_title="Recovered?",
    height=400,
)
fig.show()

### Your analysis

*TODO: Answer these questions based on your experiment:*

1. *What is the approximate detection limit (in ppm) for this star?*
2. *How does this compare to the actual depths of L 98-59's planets (400-690 ppm)?*
3. *Could ExoHunter detect an Earth-sized planet around this star? (Earth around the Sun produces ~84 ppm)*
4. *What would need to change to detect shallower transits? (Think about: noise, observation time, algorithm)*

...

---

## Exercise 8: Written report

Summarize your findings in a brief research-style report (in this cell).

### Report: Analysis of L 98-59 (TIC 261136679)

**Abstract** (2-3 sentences summarizing what you did and found):

*TODO*

**Data**: describe the light curve (cadences, time span, sectors)

*TODO*

**Methods**: briefly describe the pipeline steps you applied

*TODO*

**Results**: list each planet detected with period, depth, validation status, and catalog match

*TODO*

**Detection limits**: what is the minimum detectable depth for this star?

*TODO*

**Conclusions**: what did you learn about transit detection?

*TODO*